# 03 — Results

FR-8 deliverable artefacts (§8.3): PR-AUC and confusion matrix for all three splits, jitter degradation curve, per-tier ablation, and SHAP global importance. Reads `outputs/logs/evaluation_report.json` and `outputs/logs/decision_log.csv`, both produced by `run_pipeline.py` — run that first if these don't exist yet.

In [ ]:
import json
import pandas as pd

with open('../outputs/logs/evaluation_report.json') as f:
    report = json.load(f)

report.keys()

## Metrics across all three splits

PR-AUC is the primary metric (§8.1) — it's threshold-independent, unlike precision/recall/F1 below it, which depend on `decision.alerts_per_day_budget` / `assumed_daily_flow_volume` in `config/default.yaml`. The gap between the random split and the other two is itself a finding: a large gap means the model is memorising rather than generalising (§8.1).

In [ ]:
splits = ['random_split', 'temporal_split', 'held_out_family_split']
metrics_table = pd.DataFrame({
    split: {
        'pr_auc': report[split]['metrics']['pr_auc'],
        'precision': report[split]['metrics']['precision_at_threshold'],
        'recall': report[split]['metrics']['recall_at_threshold'],
        'f1': report[split]['metrics']['f1_at_threshold'],
        'anomaly_branch_only_pr_auc': report[split]['anomaly_branch_only_metrics']['pr_auc'],
    }
    for split in splits
}).T
metrics_table

In [ ]:
for split in splits:
    cm = report[split]['confusion_matrix']
    print(f'{split}: [[TN={cm[0][0]}, FP={cm[0][1]}], [FN={cm[1][0]}, TP={cm[1][1]}]]')

## Per-tier ablation (FR-8.8)

Quantifies the encryption-resistant floor — how much PR-AUC survives using only Tier 1 (timing) features, vs. adding Tier 2 (volume) and Tier 4 (context).

In [ ]:
pd.DataFrame(report['ablation'])

## Jitter degradation curve (FR-6.2/6.3, AC-5)

Recall against a fixed operating threshold as an attacker randomises their beacon timing by an increasing percentage. Flat or slowly-declining is good; a sharp drop means the detector relies heavily on precise periodicity and is easy to evade with jitter.

In [ ]:
jitter_df = pd.DataFrame(report['jitter_degradation_curve']) if report.get('jitter_degradation_curve') else None
jitter_df

In [ ]:
if jitter_df is not None:
    jitter_df.plot(x='jitter_pct', y='recall_at_threshold', marker='o', legend=False,
                    title='Recall vs. jitter %', ylabel='recall at fixed threshold', xlabel='jitter %')

## SHAP global feature importance (FR-7.1)

In [ ]:
shap_importance = pd.Series(report['global_shap_importance']).sort_values(ascending=False)
shap_importance.head(15)

## Sample decision log (FR-5.5, FR-7.2/7.3)

One row per scored session: risk score, both branch scores, the ALLOW/ALERT/BLOCK action, and a human-readable top-3 explanation.

In [ ]:
decision_log = pd.read_csv('../outputs/logs/decision_log.csv')
decision_log.sort_values('risk_score', ascending=False).head(10)